## Problem Statement

We’ve already extracted and chunked the research PDF:

**"Digital Transformation of the Healthcare Value Chain: Emergence of Medical Internet of Things (MIoT) may need an Integrated Clinical Environment, ICE Platform"**

We’ve also generated embeddings for each chunk using OpenAI and other models.

Now we want to:
- Store those vector embeddings in a **vector database (ChromaDB)**.
- Use the database to **query** and retrieve the most relevant chunks when a question is asked.

This step sets up the heart of the RAG pipeline: **retrieving meaningful chunks to send to the AI model.**


In [ ]:
#%run /Users/csharm33/code/genai_dbx/Course3/.setup/learner_setup.ipynb

In [ ]:
import os
import shutil
import random
import httpx
from dotenv import load_dotenv
from langchain.vectorstores import Chroma
from langchain_openai import AzureOpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyMuPDFLoader
from langchain.schema import Document
from langchain.chat_models import AzureChatOpenAI
from langchain.chains import RetrievalQA
from langchain_community.retrievers import BM25Retriever
import warnings
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
)
import openai

warnings.filterwarnings("ignore")

In [ ]:
# Authentication:
def get_access_token():
    auth = "https://api.uhg.com/oauth2/token"
    scope = "https://api.uhg.com/.default"
    grant_type = "client_credentials"


    with httpx.Client() as client:
        body = {
            "grant_type": grant_type,
            "scope": scope,
            "client_id": client_id,
            "client_secret": client_secret,
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        resp = client.post(auth, headers=headers, data=body, timeout=60)
        access_token = resp.json()["access_token"]
        return access_token


load_dotenv('/Users/csharm33/code/genai_dbx/Course3/Module2/Data/vars.env')

endpoint = os.environ.get("MODEL_ENDPOINT")
CHAT_DEPLOYMENT_NAME = os.environ.get("CHAT_DEPLOYMENT_NAME")
EMBEDDINGS_DEPLOYMENT_NAME = os.environ.get("EMBEDDINGS_MODEL_NAME")
project_id = os.environ.get("PROJECT_ID")
api_version = os.environ.get("API_VERSION")
client_id = os.environ.get("CLIENT_ID")
client_secret = os.environ.get("CLIENT_SECRET")

chat_client = openai.AzureOpenAI(
        azure_endpoint=endpoint,
        api_version=api_version,
        azure_deployment=CHAT_DEPLOYMENT_NAME,
        azure_ad_token=get_access_token(),
        default_headers={
            "projectId": project_id
        }
    )

embeddings_client = openai.AzureOpenAI(
        azure_endpoint=endpoint,
        api_version=api_version,
        azure_deployment=EMBEDDINGS_DEPLOYMENT_NAME,
        azure_ad_token=get_access_token(),
        default_headers={
            "projectId": project_id
        }
    )


In [ ]:
# this function returns vector embeddings for the provided text chunks.

@retry(wait=wait_random_exponential(min=45, max=120), stop=stop_after_attempt(6))
def get_embeddings(texts_chunk):
    return embeddings_client.embeddings.create(input=texts_chunk, model=EMBEDDINGS_DEPLOYMENT_NAME).data

In [ ]:
# this function retrieves the model's response, and returns the content of the generated message
def get_response(prompt):
    response = chat_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model=CHAT_DEPLOYMENT_NAME
    )
    
    return response.choices[0].message.content



### Define a Function to Load and Extract Text from PDF

In [ ]:
# Define a Function to Load and Extract Text from PDF
def load_pdf_with_langchain(pdf_path):
 
    # Use LangChain's built-in loader
    loader = PyMuPDFLoader(pdf_path)

    # Load the PDF into LangChain's document format
    documents = loader.load()

    print(f"Successfully loaded {len(documents)} document chunks from the PDF.")
    return documents

In [ ]:
# Path to the uploaded PDF (replace with your actual file path)
pdf_path = "/Users/csharm33/code/genai_dbx/Course3/Module2/Data/HealthcaredocforRAG.pdf"  

# Extract the document chunks
docs = load_pdf_with_langchain(pdf_path)

In [ ]:
# splitting the text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
texts = text_splitter.split_documents(docs)



### Embedding Initialization
> 
> This code initializes the `AzureOpenAIEmbeddings` model, which is integrated with ChromaDB to convert textual documents into high-dimensional vector embeddings.  
> These embeddings enable efficient semantic search, similarity matching, and retrieval operations within the vector database.


In [ ]:
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=EMBEDDINGS_DEPLOYMENT_NAME,
    api_version=api_version,
    azure_endpoint=endpoint,
    azure_ad_token=get_access_token(),
    default_headers={
        "projectId": project_id
    }
)

In [ ]:
tiktoken_cache_dir = os.path.abspath("/Users/csharm33/code/genai_dbx/Course3/.setup/tiktoken_cache/")
os.environ["TIKTOKEN_CACHE_DIR"] = tiktoken_cache_dir

In [ ]:
# Due to UHG policies, we have to disable telemetry to use ChromaDB
# See here for more information: https://docs.trychroma.com/docs/overview/telemetry
os.environ["ANONYMIZED_TELEMETRY"]="False"

# Embed and store the texts
# Supplying a persist_directory will store the embeddings on disk
persist_directory = '/tmp/vector_embeddings_OPENAI'

# Store in Chroma vector DB
vectordb = Chroma.from_documents(documents=texts, 
                            embedding=embeddings,
                            persist_directory=persist_directory)
                            
# Persist the database locally
vectordb.persist()

print("Embeddings stored in ChromaDB.")

# Data Retrieval Techniques in RAG

In a RAG (Retrieval-Augmented Generation) system, **data retrieval** is the step where we search for the most relevant information (or chunks) to a user's question — **before** giving it to the AI model to generate an answer.

### Why is it important?
- Large Language Models (LLMs) like GPT are powerful, but they **don’t know your documents** unless you tell them.
- Retrieval finds and feeds the **most relevant chunks** from your documents to the model.
- This improves **accuracy**, **relevance**, and reduces hallucinations.

In our case, we’ve already:
- Chunked the PDF data from the research paper,
- Created embeddings using BERT, Word2Vec, or OpenAI,
- Stored them using ChromaDB.

Now it’s time to **retrieve the best matches** from that database.



## Retrieval Techniques in RAG 

There are multiple ways to retrieve text chunks, but here are two major ones:

### a) Semantic Retrieval
- Uses vector similarity to find chunks **most semantically similar** to a user's question.
- Example: If someone asks "How is IoT used in healthcare?", this will match with chunks even if they don’t use the exact words — like "MIoT is transforming patient monitoring."

### b) Hybrid Retrieval
- Combines **semantic retrieval** (meaning) and **keyword search** (exact match).
- Useful when:
  - You want **precision + context**.
  - Your documents are a mix of structured and unstructured content.

## Method 1: Semantic Retrieval

In [ ]:
# Define Function to retrieve top_k semantically relevant documents from ChromaDB using vector search.
   
def semantic_retrieval(query, top_k=3):
    results = vectordb.similarity_search(query, k=top_k*2)  # fetch more to be safe
    unique_results = []
    seen_contents = set()

    for doc in results:
        if doc.page_content not in seen_contents:
            unique_results.append(doc)
            seen_contents.add(doc.page_content)
        if len(unique_results) >= top_k:
            break

    return unique_results

# Run test
query = "How does the Integrated Clinical Environment (ICE) platform support MIoT implementation in healthcare settings?"
semantic_results = semantic_retrieval(query)

# Print results
for i, doc in enumerate(semantic_results, 1):
    print(f"\n Semantic Result {i}:\n{doc.page_content}")



## Method 2: Hybrid Retrieval (BM25 + Semantic)

Hybrid retrieval combines:
- **BM25** (keyword-based scoring),
- and **vector similarity** (semantic understanding).

This gives a balanced result that captures **meaning** and **exact matches**.


In [ ]:
# Define a function to combine BM25 keyword matching with vector similarity for hybrid retrieval

def hybrid_retrieval_simple(query, top_k=3):
    """
    Combines semantic and keyword search results for diverse retrieval.
    """
    # Get semantic search results
    semantic_results = vectordb.similarity_search(query, k=top_k*2)
    semantic_contents = [doc.page_content for doc in semantic_results]
    
    # Get keyword search results
    documents = [Document(page_content=doc) if isinstance(doc, str) else doc 
                for doc in vectordb.get()["documents"]]
    bm25_retriever = BM25Retriever.from_documents(documents)
    keyword_results = bm25_retriever.get_relevant_documents(query, k=top_k*2)
    
    # Take half from semantic results
    final_results = semantic_results[:top_k//2]
    
    # Add unique keyword results
    for doc in keyword_results:
        if len(final_results) >= top_k:
            break
        if doc.page_content not in semantic_contents:
            final_results.append(doc)
    
    # Fill remaining spots with semantic results
    remaining_spots = top_k - len(final_results)
    if remaining_spots > 0:
        start_idx = len(final_results) - remaining_spots
        final_results.extend(semantic_results[start_idx:start_idx+remaining_spots])
    
    return final_results

# Test the function
hybrid_results = hybrid_retrieval_simple("How does the Integrated Clinical Environment (ICE) platform support MIoT implementation in healthcare settings?")
for i, doc in enumerate(hybrid_results, 1):
    print(f"\n Hybrid Result {i}:\n{doc.page_content}")


# What is Reranking in RAG?

In a RAG (Retrieval-Augmented Generation) system, we retrieve multiple chunks of text related to the user’s question — but:

- Not all retrieved chunks are equally relevant.

**Reranking** is the step where we:
1. Take the top N retrieved chunks,
2. Ask a smart model (like GPT) to re-evaluate and sort them,
3. Pass the most relevant chunks to the final language model for answer generation.

### Why is this important?
- Without reranking, we might feed the LLM irrelevant or less useful chunks.
- Reranking helps us **improve accuracy**, **reduce hallucination**, and **deliver more precise answers**.


## LLM-Based Reranking Using OpenAI (Semantic + LLM-Based)

This method sends the query and each chunk to OpenAI’s GPT model.

- Asks a **Language Model (like GPT)** to score or rank the relevance of each chunk.
- Understands meaning, tone, and nuance — not just keywords.
- It then asks GPT to evaluate which chunks are most relevant and return them in sorted order.


In [ ]:
# Define function to use OpenAI GPT (via ChatCompletion) to rerank document chunks based on relevance to a query.
def llm_rerank_with_openai(query, retrieved_docs, top_k=3):
    """
    Args:
        query (str): The user’s input question.
        retrieved_docs (list): List of LangChain Document objects retrieved from ChromaDB.
        top_k (int): Number of top chunks to return.

    Returns:
        list: Sorted list of the most relevant chunks, based on GPT scoring.
    """
    # Step 1: Prepare the ranking prompt
    prompt = f"You are helping rank document chunks based on how well they answer this question:\n\nQuestion: {query}\n\n"
    prompt += "Here are the chunks:\n\n"

    for i, doc in enumerate(retrieved_docs):
        prompt += f"Chunk {i+1}:\n{doc.page_content.strip()}\n\n"

    prompt += f"Please rank the top {top_k} chunks in order of relevance. Respond only like this:\nChunk 3, Chunk 1, Chunk 5"

    # Step 2: Call GPT for reranking
    
    gpt_output=get_response(prompt)
    print("GPT Rerank Output:\n", gpt_output)

    # Step 3: Extract chunk numbers from the output
    chunk_order = [int(s.strip().split()[-1]) - 1 for s in gpt_output.split(',') if s.strip().startswith("Chunk")]

    # Step 4: Return sorted chunk objects
    reranked_docs = [retrieved_docs[i] for i in chunk_order if i < len(retrieved_docs)]
    return reranked_docs


In [ ]:
# Run the reranker
reranked_results = llm_rerank_with_openai(query, hybrid_results )

# Display the reranked chunks
for i, doc in enumerate(reranked_results, 1):
    print(f"\n Reranked Chunk {i}:\n{doc.page_content[:300]}")
